In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

**exploratory data analysis**

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
# check the null in data
df.isnull().sum()

0 missing values in Every columns

In [ ]:
# drop the id columns
df.drop("customerID", axis=1, inplace=True)


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
# check the gender count and churn count
gender_counts = df['gender'].value_counts()
churn_counts = df['Churn'].value_counts()
print(gender_counts)
print(churn_counts)

In [ ]:
# Totalcharge column convert into numerical
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.dropna(inplace=True)

In [ ]:
df.info()

In [ ]:
sns.countplot(x="Churn", data = df)
plt.title("Churn Count")
plt.show()

In [ ]:
sns.countplot(x='gender', hue='Churn', data=df)
plt.title("Gender by churn")
plt.show()

In [ ]:
# tenure distribution
plt.hist(df['tenure'], bins=60)
plt.title("Tenure Distribution")
plt.xlabel("Months")
plt.ylabel("Count")
plt.show()

In [ ]:
sns.boxplot(x='Churn', y='MonthlyCharges', data=df)
plt.title("Monthly Charges vs Churn")
plt.show()

In [ ]:
sns.countplot(x='Contract', hue='Churn', data=df)
plt.title("Contract Type vs Churn")
plt.xticks(rotation=20)
plt.show()

In [ ]:
# internet service vs churn
sns.countplot(x='InternetService', hue='Churn', data=df)
plt.title("Internet Service vs Churn")
plt.xticks(rotation=20)
plt.show()

In [ ]:
# Senior Citizen vs churn
sns.countplot(x="SeniorCitizen", hue="Churn", data=df)
plt.title("Senior Citizen vs Churn")
plt.show()

In [ ]:
sns.heatmap(df.select_dtypes(include=['number']).corr())
plt.show()

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for column in df.select_dtypes(include=["object"]):
  df[column] = le.fit_transform(df[column])

In [ ]:
df.head(5)

In [ ]:
# different scale convert into same scale
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()


In [ ]:
num_cols = ["tenure", 'MonthlyCharges', 'TotalCharges']
df[num_cols] = scaler.fit_transform(df[num_cols])

In [ ]:
df.head()

Machine Learning Model build Process

In [ ]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [ ]:
print(X.shape)
print(y.shape)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


In [ ]:
linear_model = LogisticRegression()
linear_model.fit(X_train, y_train)

In [ ]:
lr_pred = linear_model.predict(X_test)

In [ ]:
lr_pred

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
print("accuracy_scor:", accuracy_score(y_test, lr_pred))
print("confusion_matrix:", confusion_matrix(y_test, lr_pred))
print("classification_report:", classification_report(y_test, lr_pred))

In [ ]:
r_model = RandomForestClassifier()
r_model.fit(X_train, y_train)

In [ ]:
r_pred = r_model.predict(X_test)

In [ ]:
print("accuracy_scor:", accuracy_score(y_test, r_pred))
print("confusion_matrix:", confusion_matrix(y_test, r_pred))
print("classification_report:", classification_report(y_test, r_pred))

Hyprypparameter fine tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs'],
    'penalty': ['l1', 'l2']
}

grid = GridSearchCV(LogisticRegression(), params, cv=5)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_

In [ ]:
print("Best Parameters:", grid.best_params_)

In [ ]:
y_pred = best_model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


In [ ]:
from sklearn.model_selection import GridSearchCV
params = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'bootstrap': [True, False]
}
grid = GridSearchCV(RandomForestClassifier(), params, cv=5)
grid.fit(X_train, y_train)
best_model = grid.best_estimator_


In [ ]:
print("Best Parameters:", grid.best_params_)

In [ ]:
y_pred = best_model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


In [ ]:
import pickle
with open("r_model.pkl", "wb") as f:
    pickle.dump(r_pred, f)